# Lookup-Only Catalog with Anonymous Write — Demo Notebook

This notebook proves a four-property authorization shape end-to-end using only
**dynamic IAM policies** + the new `/search` / `/search/catalogs/{cat}` body
filters (#819) and the core-IAM ConditionHandlers `lookup_only_search`,
`catalog_lookup_public_allowed`, and `collection_write_anonymous_allowed`
(GeoID#848 — B1 + B4 foundations).

**Use case shape:**

1. Sysadmin creates a catalog.
2. Admin configures all collections to route items via
   `items_elasticsearch_private_driver`.
3. Admin creates two collections (`coll1`, `coll2`).
4. Admin uses dynamic policies to deny every service on this catalog except:
   * `POST /features/catalogs/{cat}/collections/{coll1,coll2}/items` — open
     for anonymous writes.
   * `/search` and `/search/catalogs/{cat}` — open only when the request
     carries a `geoid` or `external_id` filter (no enumeration).
5. Anonymous clients write 2 items per collection (4 items total).
6. Nobody can list / navigate / read any of those items.
7. Anyone who knows the `geoid` or `external_id` can retrieve the item via
   `/search` or `/search/catalogs/{cat}`.

**Run target:** a review env that does **not** require authentication
(so the demo can bypass token-quota issues during development). Once
verified, the same notebook re-runs against the auth-enforcing env after
a token plumb step (operator-driven, outside this notebook).

**Aliasing & privacy:** all identifiers are env-driven. The notebook
hard-codes nothing about the target deployment.


## 0. Environment bootstrap

Read config from environment / `.env`. Three clients: sysadmin, admin, anonymous.

In [ ]:
import os
import time
import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
DEMO_CATALOG_ID = os.environ.get("DEMO_CATALOG_ID", "lookup-demo")
COLL_IDS = ("coll1", "coll2")

# Tokens are optional in the no-auth review env. When set they unlock
# sysadmin + admin operations on auth-enforcing envs.
SYSADMIN_TOKEN = os.environ.get("DYNASTORE_SYSADMIN_TOKEN", "")
ADMIN_TOKEN = os.environ.get("DYNASTORE_ADMIN_TOKEN", SYSADMIN_TOKEN)

def _client(token: str = "") -> httpx.Client:
    headers = {"Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    return httpx.Client(base_url=BASE_URL, headers=headers, timeout=30.0)

sysadmin = _client(SYSADMIN_TOKEN)
admin = _client(ADMIN_TOKEN)
anon = _client()  # explicitly no Authorization

print(f"BASE_URL     = {BASE_URL}")
print(f"CATALOG      = {DEMO_CATALOG_ID}")
print(f"COLLECTIONS  = {COLL_IDS}")
print(f"AUTH MODE    = {'auth-on' if SYSADMIN_TOKEN else 'anonymous (review env without IDP)'}")


## 1. Sysadmin: create the catalog

Create a fresh catalog. Idempotent on re-run — if it already exists, surface
the existing record and move on.


In [ ]:
def create_catalog(cat_id: str) -> dict:
    """Create the catalog or surface the existing one."""
    body = {
        "id": cat_id,
        "type": "Catalog",
        "title": "Lookup-only demo catalog",
        "description": (
            "Anonymous-write + lookup-only-search demo. Items in this catalog "
            "are not enumerable via STAC/Features/Coverages/etc. — they are "
            "retrievable only by their GeoID or external_id."
        ),
        "stac_version": "1.0.0",
    }
    r = sysadmin.post("/stac/catalogs", json=body)
    if r.status_code in (200, 201):
        return r.json()
    if r.status_code == 409:
        print(f"catalog '{cat_id}' already exists — reusing")
        r = sysadmin.get(f"/stac/catalogs/{cat_id}")
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

cat = create_catalog(DEMO_CATALOG_ID)
print({k: cat.get(k) for k in ("id", "type", "title")})


Wait for the catalog to be addressable (catalog provisioning is async).

In [ ]:
def _is_still_provisioning_409(resp: httpx.Response) -> bool:
    """True iff a 409 carries the platform's 'still provisioning' detail.

    The catalog provisioning hand-off is async: POST /stac/catalogs returns
    201 immediately and a background job builds the per-catalog schema +
    seeds IAM. Until that job finishes, mutations under the catalog (e.g.
    create-collection) reject with 409 + detail containing
    'still provisioning'. Distinguishing this from a genuine
    'resource already exists' 409 prevents misleading 404 fallbacks.
    """
    if resp.status_code != 409:
        return False
    try:
        detail = resp.json().get("detail", "")
    except Exception:
        return False
    return isinstance(detail, str) and "still provisioning" in detail.lower()

def wait_for_catalog(cat_id: str, timeout_s: int = 30) -> None:
    """Wait until GET /stac/catalogs/{id} returns 200.

    This only confirms the catalog row is reachable — async provisioning
    (schema build + IAM seed) typically takes 2-5 minutes longer and is
    NOT surfaced on the public STAC view. The provisioning-aware retry
    lives in `create_collection`; that's where the wait actually happens
    on a freshly created catalog.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        r = sysadmin.get(f"/stac/catalogs/{cat_id}")
        if r.status_code == 200:
            return
        time.sleep(0.5)
    raise TimeoutError(f"catalog {cat_id} did not become reachable within {timeout_s}s")

wait_for_catalog(DEMO_CATALOG_ID)
print(f"catalog '{DEMO_CATALOG_ID}' URL reachable (background provisioning may still be in flight)")


## 2. Admin: collections + private items routing

For each collection:

1. Create the collection.
2. PUT `CollectionPrivacy.is_private=true` (satisfies the cascade gate).
3. PUT `ItemsRoutingConfig` pinning `items_elasticsearch_private_driver` in
   WRITE / READ / SEARCH. WRITE also fans to the postgres driver so the
   items survive engine restarts.
4. PUT `CollectionWriteAudience.allow_anonymous_create=true` — the runtime
   knob the anonymous-write policy condition reads.


In [ ]:
def create_collection(
    cat_id: str,
    coll_id: str,
    *,
    provisioning_timeout_s: int = 300,
    poll_interval_s: float = 2.0,
) -> dict:
    """Create a collection, retrying while the parent catalog is still
    provisioning. Falls back to GET only on a genuine 'already exists' 409.

    `provisioning_timeout_s` defaults to 5 min — the upper end of the
    typical async-provisioning window. Bump it for slow review envs.
    """
    body = {
        "id": coll_id,
        "type": "Collection",
        "title": f"Demo collection {coll_id}",
        "description": "Demo collection accepting anonymous item writes.",
        "stac_version": "1.0.0",
        "license": "proprietary",
        "extent": {
            "spatial": {"bbox": [[-180.0, -90.0, 180.0, 90.0]]},
            "temporal": {"interval": [[None, None]]},
        },
    }
    deadline = time.monotonic() + provisioning_timeout_s
    attempts = 0
    while True:
        attempts += 1
        r = admin.post(f"/stac/catalogs/{cat_id}/collections", json=body)
        if r.status_code in (200, 201):
            if attempts > 1:
                print(f"  collection '{coll_id}' created after {attempts} attempts "
                      f"(catalog provisioning settled)")
            return r.json()
        if _is_still_provisioning_409(r):
            if time.monotonic() >= deadline:
                raise TimeoutError(
                    f"catalog '{cat_id}' still provisioning after "
                    f"{provisioning_timeout_s}s ({attempts} attempts) — "
                    f"last detail: {r.json().get('detail', '')!r}"
                )
            time.sleep(poll_interval_s)
            continue
        if r.status_code == 409:
            print(f"  collection '{coll_id}' already exists — reusing")
            r = admin.get(f"/stac/catalogs/{cat_id}/collections/{coll_id}")
            r.raise_for_status()
            return r.json()
        r.raise_for_status()

def put_config(cat_id: str, coll_id: str, plugin_id: str, config_body: dict) -> dict:
    url = f"/configs/catalogs/{cat_id}/collections/{coll_id}/plugins/{plugin_id}"
    r = admin.put(url, json=config_body)
    r.raise_for_status()
    return r.json() if r.text else {}

# Routing config: WRITE + READ + SEARCH all pin the private items driver.
# WRITE additionally fans to postgres so items are durable beyond ES.
ROUTING_CONFIG = {
    "operations": {
        "WRITE": [
            {"driver_ref": "items_postgresql_driver", "on_failure": "fatal"},
            {"driver_ref": "items_elasticsearch_private_driver",
             "write_mode": "async", "on_failure": "outbox"},
        ],
        "READ": [
            {"driver_ref": "items_elasticsearch_private_driver",
             "hints": ["geometry_simplified"], "on_failure": "warn"},
            {"driver_ref": "items_postgresql_driver",
             "hints": ["geometry_exact"], "on_failure": "fatal"},
        ],
        "SEARCH": [
            {"driver_ref": "items_elasticsearch_private_driver",
             "hints": ["geometry_simplified"], "on_failure": "warn"},
            {"driver_ref": "items_postgresql_driver",
             "hints": ["geometry_exact"], "on_failure": "fatal"},
        ],
    },
}

# Items PG driver config: enable external_id so the `_external_id` field
# is populated from `properties.external_id` (the field the lookup
# matcher reads on /search).
ITEMS_PG_DRIVER_CONFIG = {
    "engine_ref": "postgresql_engine_config",
    "sidecars": [
        {"sidecar_type": "geometries"},
        {"sidecar_type": "attributes",
         "enable_external_id": True,
         "external_id_field": "properties.external_id"},
    ],
}

for coll in COLL_IDS:
    print(f"-- collection '{coll}' --")
    create_collection(DEMO_CATALOG_ID, coll)
    put_config(DEMO_CATALOG_ID, coll, "collection_privacy", {"is_private": True})
    put_config(DEMO_CATALOG_ID, coll, "items_postgresql_driver_config", ITEMS_PG_DRIVER_CONFIG)
    put_config(DEMO_CATALOG_ID, coll, "items_routing_config", ROUTING_CONFIG)
    put_config(DEMO_CATALOG_ID, coll, "collection_write_audience",
               {"allow_anonymous_create": True})
    print(f"  -> privacy=on, private-items routing pinned, anonymous-write enabled")


In [ ]:
# Catalog-level: opt-in to anonymous lookup access (used by the lookup
# ALLOW policy in section 3 alongside the `lookup_only_search` condition).
admin.put(
    f"/configs/catalogs/{DEMO_CATALOG_ID}/plugins/catalog_lookup_audience",
    json={"is_public": True},
).raise_for_status()
print(f"catalog '{DEMO_CATALOG_ID}' opted in to anonymous lookup access")


## 3. Admin: catalog-scoped policy bundle + bind to anonymous role

Two sub-steps:

**3a. POST the policy bundle** to `/admin/policies?catalog_id={cat}`:
- A **DENY-all** block for every OGC + platform service prefix at this catalog.
- **3 ALLOW carve-outs**:
  - Anonymous `POST /features/catalogs/{cat}/collections/(coll1|coll2)/items` — gated by `collection_write_anonymous_allowed`.
  - Anonymous `GET/POST /search/catalogs/{cat}` — gated by `lookup_only_search` + `catalog_lookup_public_allowed`.
  - Anonymous `GET/POST /search` (unscoped, cross-catalog needle) — same conditions.

**3b. Bind the ALLOW policy IDs to the `unauthenticated` role at this catalog's scope.**
Anonymous traffic only evaluates policies attached to that role; without the binding step
the ALLOW carve-outs are present in storage but never reached.

### Platform caveat (dynastore#291)

The `PermissionService.evaluate_access` function used for role-based policies
is **first-match-wins** (iterates `effective_policies` and returns on the first
matching DENY *or* ALLOW). It is NOT deny-precedence today; the sibling
`evaluate_policy_statements` (used for principal-attached custom policies) is.

Consequence for this demo: the DENY-all bundle in 3a only overrides global
ALLOWs (e.g. `features_public_access`, `stac_public_access`) when its policies
happen to iterate before them. Iteration order is `set`-derived → not stable
across runs. Until **dynastore#291** is fixed, the deny-matrix in section 5 is
**best-effort observability**, not deterministic. The lookup-matrix in section 6
remains deterministic because the carve-out ALLOWs are evaluated on a path
that no global ALLOW covers.


In [ ]:
# Every OGC + platform-tier service that exposes catalog-scoped URLs.
SERVICE_PREFIXES = [
    "stac", "features", "coverages", "records", "tiles", "processes",
    "edr", "maps", "styles", "dggs", "consys", "moving_features",
]

def deny_all_except_lookup(cat_id: str) -> list[dict]:
    policies: list[dict] = []

    # 1. DENY everything at catalog scope.
    for svc in SERVICE_PREFIXES:
        policies.append({
            "id": f"{cat_id}-deny-{svc}",
            "effect": "DENY",
            "actions": ["GET", "POST", "PUT", "PATCH", "DELETE"],
            "resources": [f"^/{svc}/catalogs/{cat_id}(/.*)?$"],
        })

    # 2. ALLOW anonymous POST to specific collections via Features API.
    feature_resources = [
        f"^/features/catalogs/{cat_id}/collections/{coll}/items/?$"
        for coll in COLL_IDS
    ]
    policies.append({
        "id": f"{cat_id}-allow-anon-features-write",
        "effect": "ALLOW",
        "actions": ["POST"],
        "resources": feature_resources,
        "conditions": [{"type": "collection_write_anonymous_allowed"}],
    })

    # 3. ALLOW anonymous GET/POST on /search* — gated by lookup_only_search
    #    so the request MUST carry geoid or external_id and MUST NOT carry
    #    a broadening filter (bbox, intersects, datetime, filter, q).
    #    Also gated by catalog_lookup_public_allowed (the per-catalog opt-in).
    policies.append({
        "id": f"{cat_id}-allow-anon-search-lookup-scoped",
        "effect": "ALLOW",
        "actions": ["GET", "POST"],
        "resources": [f"^/search/catalogs/{cat_id}/?$"],
        "conditions": [
            {"type": "lookup_only_search"},
            {"type": "catalog_lookup_public_allowed"},
        ],
    })
    # The unscoped /search is also opened so cross-catalog needle lookup
    # works for callers who know a GeoID but not its home catalog. This
    # policy is scoped to this catalog's tenant data via the lookup
    # condition + the SEARCH driver routing isolating ES indices.
    policies.append({
        "id": f"{cat_id}-allow-anon-search-lookup-unscoped",
        "effect": "ALLOW",
        "actions": ["GET", "POST"],
        "resources": [r"^/search/?$"],
        "conditions": [
            {"type": "lookup_only_search"},
            {"type": "catalog_lookup_public_allowed"},
        ],
    })

    return policies

POLICY_BUNDLE = deny_all_except_lookup(DEMO_CATALOG_ID)

print(f"Policy bundle: {len(POLICY_BUNDLE)} policies")
for p in POLICY_BUNDLE:
    cond = ""
    if p.get("conditions"):
        cond = " conditions=[" + ", ".join(c["type"] for c in p["conditions"]) + "]"
    print(f"  {p['effect']:5} {','.join(p['actions']):20} {p['resources']}{cond}")


In [ ]:
def upsert_policy(cat_id: str, policy: dict) -> dict:
    r = admin.post(f"/admin/policies?catalog_id={cat_id}", json=policy)
    if r.status_code in (200, 201):
        return r.json()
    if r.status_code == 409:
        # Idempotent re-run — overwrite the existing record.
        r = admin.put(f"/admin/policies/{policy['id']}?catalog_id={cat_id}", json=policy)
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

for p in POLICY_BUNDLE:
    upsert_policy(DEMO_CATALOG_ID, p)
print(f"posted {len(POLICY_BUNDLE)} policies for catalog '{DEMO_CATALOG_ID}'")


In [ ]:
# 3b. Bind the ALLOW policy IDs to the `unauthenticated` role at this
# catalog's scope. Anonymous traffic only evaluates policies attached to
# that role; without this step the ALLOW carve-outs in 3a are present in
# storage but `evaluate_access` never reaches them.
#
# Pattern (idempotent): fetch the catalog-scoped `unauthenticated` role
# if it exists, otherwise start from the empty list, then PUT back with
# our ALLOW policy IDs appended (deduplicated).
ANON_ROLE = "unauthenticated"  # IamRolesConfig().anonymous_role_name default

ALLOW_POLICY_IDS = [p["id"] for p in POLICY_BUNDLE if p["effect"] == "ALLOW"]
DENY_POLICY_IDS = [p["id"] for p in POLICY_BUNDLE if p["effect"] == "DENY"]
# We bind BOTH ALLOW and DENY policies to the role — the DENYs serve the
# defence-in-depth purpose noted in the section 3 platform caveat (best-
# effort enforcement against global *_public_access ALLOWs, deterministic
# once dynastore#291 is fixed).
TO_BIND = ALLOW_POLICY_IDS + DENY_POLICY_IDS

# Fetch existing catalog-scoped role (404 -> start empty).
r = admin.get(f"/admin/roles/{ANON_ROLE}", params={"catalog_id": DEMO_CATALOG_ID})
if r.status_code == 200:
    existing = r.json().get("policies") or []
else:
    existing = []

merged = list(dict.fromkeys(existing + TO_BIND))  # preserve order, dedupe

# PUT back to upsert the catalog-scoped role variant.
update_body = {"policies": merged}
r = admin.put(
    f"/admin/roles/{ANON_ROLE}",
    params={"catalog_id": DEMO_CATALOG_ID},
    json=update_body,
)
if r.status_code == 404:
    # Catalog-scoped variant doesn't exist yet — create via POST.
    create_body = {
        "name": ANON_ROLE,
        "description": (
            f"Catalog-scoped variant of '{ANON_ROLE}' for {DEMO_CATALOG_ID} — "
            "carries the demo's ALLOW + DENY carve-outs alongside global "
            "anonymous policies."
        ),
        "policies": merged,
    }
    r = admin.post(
        "/admin/roles",
        params={"catalog_id": DEMO_CATALOG_ID},
        json=create_body,
    )
r.raise_for_status()
print(
    f"bound {len(TO_BIND)} policies to role '{ANON_ROLE}' "
    f"(catalog_id={DEMO_CATALOG_ID}): "
    f"{len(ALLOW_POLICY_IDS)} ALLOW + {len(DENY_POLICY_IDS)} DENY"
)


## 4. Anonymous client: POST 2 items per collection

Items carry deterministic `properties.geoid` and `properties.external_id` so
the lookup probe in section 6 has known needles.


In [ ]:
def make_item(coll_id: str, idx: int) -> dict:
    geoid = f"demo-{coll_id}-{idx:02d}"
    external_id = f"ext-{coll_id}-{idx:02d}"
    return {
        "type": "Feature",
        "id": geoid,  # the OGC Features id; the geoid extension stores it
        "stac_version": "1.0.0",
        "geometry": {"type": "Point", "coordinates": [10.0 + idx, 45.0 + idx]},
        "properties": {
            "datetime": "2026-01-01T00:00:00Z",
            "geoid": geoid,
            "external_id": external_id,
            "description": f"Demo item {idx} in {coll_id}",
        },
        "links": [],
        "assets": {},
    }

written = []  # list of (coll_id, geoid, external_id)
for coll in COLL_IDS:
    for idx in (1, 2):
        item = make_item(coll, idx)
        r = anon.post(
            f"/features/catalogs/{DEMO_CATALOG_ID}/collections/{coll}/items",
            json=item,
        )
        if r.status_code not in (200, 201):
            print(f"WRITE {coll}/{idx}: HTTP {r.status_code}: {r.text[:200]}")
        r.raise_for_status()
        written.append((coll, item["properties"]["geoid"], item["properties"]["external_id"]))
        print(f"  wrote {coll}/{item['properties']['geoid']}")

print(f"\ntotal items written anonymously: {len(written)}")


## 5. Anonymous client: deny matrix probe (best-effort observability)

For every service prefix, an anonymous probe against the catalog-scoped path
should be denied (401 or 403). On platforms where dynastore#291 is **not yet
fixed**, the verdict depends on whether the catalog-scoped DENY happens to
iterate before the global `*_public_access` ALLOW — non-deterministic. Treat
this section as observability, not assertion-based, until #291 lands. The
lookup-matrix in section 6 is deterministic regardless.


In [ ]:
DENY_PROBES = []
for svc in SERVICE_PREFIXES:
    DENY_PROBES.append(("GET", f"/{svc}/catalogs/{DEMO_CATALOG_ID}", None))
    DENY_PROBES.append(("GET", f"/{svc}/catalogs/{DEMO_CATALOG_ID}/collections", None))

# Also probe a known item URL via Features (the catalog-scoped read path).
for coll, geoid, _ in written:
    DENY_PROBES.append((
        "GET",
        f"/features/catalogs/{DEMO_CATALOG_ID}/collections/{coll}/items/{geoid}",
        None,
    ))

denied = 0
allowed = []  # surprise allows
for method, url, body in DENY_PROBES:
    if body is None:
        r = anon.request(method, url)
    else:
        r = anon.request(method, url, json=body)
    if r.status_code in (401, 403):
        denied += 1
    else:
        allowed.append((method, url, r.status_code))

print(f"deny matrix: {denied}/{len(DENY_PROBES)} denied as expected")
if allowed:
    print("\nUNEXPECTED allows (investigate):")
    for m, u, s in allowed:
        print(f"  {m} {u} -> {s}")
else:
    print("zero surprise allows — deny matrix clean")
if allowed:
    print("\nNote: unexpected ALLOWs likely reflect dynastore#291 "
          "(first-match-wins vs deny-precedence). Lookup-matrix in "
          "section 6 is unaffected.")


## 6. Anonymous client: lookup matrix probe

* `POST /search/catalogs/{cat}` with `geoid` body filter → 200 returning the item.
* `POST /search/catalogs/{cat}` with `external_id` body filter → 200 returning the item.
* `POST /search` (unscoped) with `geoid` body filter → 200 (cross-catalog needle).
* `POST /search/catalogs/{cat}` with `bbox` only → 403 (no lookup field).
* `GET /search/catalogs/{cat}?geoid=...` → 200.
* `GET /search/catalogs/{cat}` (no filter) → 403.


In [ ]:
def post_search(scoped: bool, body: dict) -> tuple[int, dict]:
    path = f"/search/catalogs/{DEMO_CATALOG_ID}" if scoped else "/search"
    r = anon.post(path, json=body)
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, {"_raw": r.text[:200]}

def get_search(query: dict) -> tuple[int, dict]:
    r = anon.get(f"/search/catalogs/{DEMO_CATALOG_ID}", params=query)
    try:
        return r.status_code, r.json()
    except Exception:
        return r.status_code, {"_raw": r.text[:200]}

# Pick the first known needle.
sample_coll, sample_geoid, sample_external_id = written[0]

probes = [
    ("POST /search/catalogs scoped, geoid",
     lambda: post_search(True, {"geoid": [sample_geoid]}),
     200),
    ("POST /search/catalogs scoped, external_id",
     lambda: post_search(True, {"external_id": [sample_external_id]}),
     200),
    ("POST /search unscoped, geoid (cross-catalog needle)",
     lambda: post_search(False, {"geoid": [sample_geoid]}),
     200),
    ("POST /search/catalogs scoped, bbox only (broadening — must deny)",
     lambda: post_search(True, {"bbox": [-180, -90, 180, 90]}),
     403),
    ("GET /search/catalogs?geoid=... (scoped lookup)",
     lambda: get_search({"geoid": sample_geoid}),
     200),
    ("GET /search/catalogs (no filter — must deny)",
     lambda: get_search({}),
     403),
]

ok = 0
for label, fn, expected in probes:
    status, body = fn()
    verdict = "OK" if status == expected else "MISMATCH"
    print(f"[{verdict:8}] expected {expected}, got {status}  {label}")
    if status == expected:
        ok += 1
    if status == 200 and isinstance(body, dict):
        n = len(body.get("features", body.get("items", [])))
        print(f"           returned {n} item(s)")

print(f"\nlookup matrix: {ok}/{len(probes)} matched expected verdict")


## 7. Cleanup

Drop the demo state so the notebook can be re-run from a clean slate.

Run this only when finished — the operator-side runbook may want the catalog
to persist for ongoing review.


In [ ]:
def cleanup() -> None:
    # Remove policies first so admin auth on the catalog still works.
    for p in POLICY_BUNDLE:
        try:
            admin.delete(f"/admin/policies/{p['id']}?catalog_id={DEMO_CATALOG_ID}")
        except Exception:
            pass

    # Turn off the audience opt-ins.
    try:
        admin.put(
            f"/configs/catalogs/{DEMO_CATALOG_ID}/plugins/catalog_lookup_audience",
            json={"is_public": False},
        )
    except Exception:
        pass
    for coll in COLL_IDS:
        try:
            admin.put(
                f"/configs/catalogs/{DEMO_CATALOG_ID}/collections/{coll}/plugins/collection_write_audience",
                json={"allow_anonymous_create": False},
            )
        except Exception:
            pass
        # Toggle privacy off to drop the cascade-pinned routing requirement,
        # then delete collection. Actual deletion left to operator — the
        # cascade-validator may reject toggling privacy=False if the routing
        # still pins the private driver; operator decides whether to teardown.

    print("cleanup: policies + audience opt-ins reverted. "
          "Collection / catalog deletion left to operator discretion.")

# Uncomment to run:
# cleanup()
print("(cleanup is opt-in — un-comment the call above to execute)")


---

## Tracking & follow-ups

Implementation of the underlying handlers (`lookup_only_search`,
`catalog_lookup_public_allowed`, `collection_write_anonymous_allowed`)
landed in **GeoID#848**. Once verified end-to-end against the review
env using this notebook, the geoid extension itself is a candidate for
removal — all its anonymous-flow capabilities are now operator-PUT'd
via dynamic policies and core IAM, with no extension dependency.
